# GraphRAG Neo4j Cypher Integration Demo

This notebook demonstrates the experimental Neo4j integration for GraphRAG:

1. **Indexing files** - Standard GraphRAG indexing
2. **Writing to Neo4j** - Automatic graph persistence (feature-flagged)
3. **Cypher queries** - Direct graph queries for retrieval

## EXPERIMENTAL

This integration is experimental and feature-flagged. All Neo4j functionality is controlled by the `GRAPHRAG_USE_NEO4J` environment variable.

## Schema

The minimal graph schema used:

- **Nodes**:
  - `(:Document {id, text, source})`
  - `(:Entity {id, name, type})`
- **Relationships**:
  - `(:Document)-[:MENTIONS]->(:Entity)`


## Setup


In [ ]:
import os

# Enable Neo4j integration (EXPERIMENTAL)
os.environ["GRAPHRAG_USE_NEO4J"] = "true"

# Neo4j connection details
os.environ["NEO4J_URI"] = "bolt://localhost:7687"  # or neo4j+s://xxxx.databases.neo4j.io
os.environ["NEO4J_USER"] = "neo4j"
os.environ["NEO4J_PASSWORD"] = "your_password"  # Replace with your password


## Step 1: Index Documents

Run standard GraphRAG indexing. When `GRAPHRAG_USE_NEO4J=true`, the indexing pipeline will automatically write documents, entities, and MENTIONS relationships to Neo4j after the `create_final_documents` workflow completes.


In [ ]:
from graphrag.api.index import build_index
from graphrag.config.load_config import load_config
import asyncio

# Load your GraphRAG config
config = load_config("path/to/your/config.yaml")

# Run indexing (Neo4j writes happen automatically if enabled)
results = asyncio.run(build_index(config, verbose=True))

print("Indexing complete!")
print(f"If Neo4j is enabled, data has been written to Neo4j.")


## Step 2: Query Neo4j with Cypher

Use the `run_cypher` utility function to run arbitrary Cypher queries. This is intended for exploration and notebook use only.


In [ ]:
from graphrag.graph.neo4j_client import run_cypher

# Example 1: Get all entities
entities = run_cypher("MATCH (e:Entity) RETURN e.id as id, e.name as name, e.type as type LIMIT 10")
print("Sample entities:")
for entity in entities:
    print(f"  - {entity['name']} ({entity['type']})")


In [ ]:
# Example 2: Get documents mentioning a specific entity
entity_name = "Microsoft"  # Replace with an entity from your data
docs = run_cypher(
    "MATCH (d:Document)-[:MENTIONS]->(e:Entity {name: $name}) RETURN d.id as id, d.text as text, d.source as source LIMIT 5",
    params={"name": entity_name}
)
print(f"Documents mentioning '{entity_name}':")
for doc in docs:
    print(f"  - {doc['source']}: {doc['text'][:100]}...")


In [ ]:
# Example 3: Count documents per entity
counts = run_cypher(
    "MATCH (d:Document)-[:MENTIONS]->(e:Entity) "
    "RETURN e.name as entity_name, e.type as entity_type, count(d) as doc_count "
    "ORDER BY doc_count DESC LIMIT 10"
)
print("Top entities by document count:")
for row in counts:
    print(f"  - {row['entity_name']} ({row['entity_type']}): {row['doc_count']} documents")


## Step 3: Validation

Verify that data was written correctly to Neo4j:


In [ ]:
# Count nodes and relationships
stats = run_cypher("""
    MATCH (d:Document)
    WITH count(d) as doc_count
    MATCH (e:Entity)
    WITH doc_count, count(e) as entity_count
    MATCH ()-[r:MENTIONS]->()
    RETURN doc_count, entity_count, count(r) as mention_count
""")

if stats:
    s = stats[0]
    print(f"Documents: {s['doc_count']}")
    print(f"Entities: {s['entity_count']}")
    print(f"MENTIONS relationships: {s['mention_count']}")
else:
    print("No data found in Neo4j. Make sure indexing completed with Neo4j enabled.")
